# 03 — Ablation Study

Uses the frozen full-SAR configuration. We do not retune the ablations, because doing so would make signal-removal comparisons confounded by different validation-selected hyperparameters. Question-Only uses the original question inside SAR; Sentence Transformer Search remains a separate baseline using the current MIGRES query set.

Variants: SAR-Full, SAR-Without-Relevance, SAR-Without-Gap, SAR-Without-Evidence, SAR-Without-Tuning, Question-Only.

In [ ]:
from pathlib import Path
import sys, json, yaml, platform
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print("Project root:", ROOT)
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Selected device:", DEVICE)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    DEVICE = "cpu"
    print("Torch/GPU probe unavailable:", type(e).__name__, e)
PLAN = yaml.safe_load((ROOT / "configs" / "experiments.yaml").read_text())
BASE_CONFIG = [ROOT / "configs" / "default.yaml", ROOT / "configs" / "gemini.yaml"]


In [ ]:
RUN_EXPERIMENT=False
LIMIT_PER_DATASET=None
OUT_ROOT=ROOT/'outputs/experiments/03_ablation'
FROZEN_PATH=ROOT/'outputs/experiments/01_parameter_selection/selected_config.json'


In [ ]:
from sar_migres.utils.config import load_config
from sar_migres.datasets.loader import load_phase_questions
from sar_migres.evaluation.aggregation import AblationStudy
from sar_migres.experiments.parameter_selection import frozen_setting
from sar_migres.experiments.registry import ablation_overrides
from sar_migres.experiments.runner import SARMigresPipeline
from sar_migres.experiments.iterative import answer_metrics_from_run
from sar_migres.evaluation.plotting import save_bar_plot
from sar_migres.utils.paths import make_run_id
RUN_DIRS_BY_DATASET={}
base=load_config(*BASE_CONFIG)
from sar_migres.utils.config import validate_canonical_protocol
validate_canonical_protocol(base)
print('Frozen parameters:', 'present' if FROZEN_PATH.exists() else 'MISSING (required before RUN_EXPERIMENT=True)')
print('Retune:',PLAN['ablations']['retune_weights'])

if not RUN_EXPERIMENT:
    print('Preflight only. Final split:', base.data.final_split)
else:
    k,w=frozen_setting(FROZEN_PATH)
    variants=AblationStudy.ablation_weights(w)
    print('Frozen K/weights:',k,w.as_tuple())
    for ds in base.data.datasets:
        RUN_DIRS_BY_DATASET[ds]={}
        items=load_phase_questions(ds,base.data.dataset_paths[ds],'final',limit=LIMIT_PER_DATASET,final_split=base.data.final_split,validation_fraction=base.data.validation_fraction,validation_seed=base.data.validation_seed)
        for name,weights in variants.items():
            ov=ablation_overrides(name,weights,k=k,run_name=f'{ds}_{name.lower().replace(" ","_")}')
            ov['data']={'dataset':ds,'dataset_path':base.data.dataset_paths[ds],'split':base.data.final_split}
            ov['run']={'name':f'{ds}_{name.lower().replace(" ","_")}', 'output_dir':str(OUT_ROOT),'resume':False}
            run_id=make_run_id(f'{ds}_{name}')
            run_dir=OUT_ROOT/name/'runs'/run_id
            pipe=SARMigresPipeline(load_config(*BASE_CONFIG,overrides=ov),run_id=run_id,run_dir=run_dir).prepare()
            pipe.run_batch(items,resume=False)
            RUN_DIRS_BY_DATASET[ds][name]=pipe.run_dir
            print(ds,name,'->',pipe.run_dir)


In [ ]:
PLOT_ROOT=OUT_ROOT/'plots'
for ds,runs in RUN_DIRS_BY_DATASET.items():
    if not runs: continue
    save_bar_plot(list(runs),[answer_metrics_from_run(path).get('exact_match',float('nan')) for path in runs.values()],
                  'Exact match','Ablation comparison: '+ds,PLOT_ROOT/f'{ds}_exact_match.png')
print('Ablation plots:',PLOT_ROOT)
